# Employetics — 40s Complete Motion Graphics Video Ad (Google Colab GPU)

Render the full **40-second, 1080p / 4K @ 60fps** SaaS motion-graphics commercial for **Employetics** with real NVIDIA GPU acceleration:
- **Chrome for Testing + NVIDIA EGL (`--chrome-mode=chrome-for-testing --gl=angle-egl`)** for GPU-accelerated frame compositing
- **Kokoro-82M CUDA GPU Neural Voiceover** (with automatic fallback to Edge-TTS / gTTS / espeak-ng so audio is always present)
- **Custom Procedural Modern SaaS Soundtrack + Synchronized UI SFX** with automatic voiceover sidechain ducking
- **Direct NVIDIA NVENC GPU Video Encoding (`h264_nvenc`)** + guaranteed AAC 256kbps stereo audio muxing & loudness verification

> **Before running:** Go to **Runtime → Change runtime type** and select **T4 GPU** (or **L4 / A100 GPU**), then click **Runtime → Run all**.

In [ ]:
#@title 1. Configure Your Render Settings
GITHUB_REPO_URL = "https://github.com/abdullahmehmood1/EmployeticsAD.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
TTS_ENGINE = "auto" #@param ["auto", "kokoro", "edge", "gtts"]
KOKORO_VOICE = "am_michael" #@param ["am_michael", "am_fenrir", "am_adam", "af_heart", "af_bella", "bm_george"]
VOICE_SPEED = 1.02 #@param {type:"slider", min:0.85, max:1.20, step:0.01}
MUSIC_VOLUME = 0.72 #@param {type:"slider", min:0.0, max:1.0, step:0.02}
BURN_CAPTIONS = False #@param {type:"boolean"}
RESOLUTION = "1080p (1920x1080 @ 60fps)" #@param ["1080p (1920x1080 @ 60fps)", "4K (3840x2160 @ 60fps)"]
SAVE_TO_GOOGLE_DRIVE = False #@param {type:"boolean"}

SCALE = 2.0 if RESOLUTION.startswith("4K") else 1.0
OUTPUT_FILENAME = "employetics-ad-4k60.mp4" if SCALE >= 2.0 else "employetics-ad-1080p60.mp4"
print(f"Configured: {RESOLUTION} | Engine={TTS_ENGINE} | Voice={KOKORO_VOICE} | Captions={BURN_CAPTIONS}")


In [ ]:
#@title 2. Verify NVIDIA GPU, Register EGL Drivers & Install Dependencies
import json, os, pathlib, shutil, subprocess

print("=== 1) Checking NVIDIA GPU ===")
gpu_check = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True)
if gpu_check.returncode == 0:
    print("GPU ACTIVE:", gpu_check.stdout.strip())
else:
    print("WARNING: No NVIDIA GPU detected. Switch Runtime -> Change runtime type -> T4 GPU!")

print("\n=== 2) Installing system GPU/EGL/Chrome/Audio packages ===")
!apt-get -qq update
!apt-get -qq install -y ffmpeg espeak-ng xvfb libglvnd0 libegl1 libgl1 libgles2 libvulkan1 vulkan-tools mesa-utils libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libasound2 libpango-1.0-0 libcairo2

# Register NVIDIA GLVND EGL ICD so Chrome EGL binds to NVIDIA T4/L4/A100 instead of CPU Mesa
egl_vendor = pathlib.Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
if not egl_vendor.exists():
    for d in ["/usr/lib/x86_64-linux-gnu", "/usr/local/nvidia/lib64", "/usr/lib64"]:
        cand = pathlib.Path(d) / "libEGL_nvidia.so.0"
        if cand.exists():
            egl_vendor.parent.mkdir(parents=True, exist_ok=True)
            egl_vendor.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": str(cand)}}, indent=2))
            print(f"Registered NVIDIA EGL ICD -> {egl_vendor} ({cand})")
            break

print("\n=== 3) Installing Node.js 22 ===")
node_ver = subprocess.run(["node", "-v"], capture_output=True, text=True)
if node_ver.returncode != 0 or not node_ver.stdout.strip().startswith(("v20", "v22", "v24")):
    !curl -fsSL https://deb.nodesource.com/setup_22.x | bash - >/dev/null 2>&1
    !apt-get -qq install -y nodejs
!node -v && npm -v

print("\n=== 4) Cloning / Updating Project Repository ===")
PROJECT_DIR = "/content/employeticsAD"
if os.path.exists(os.path.join(PROJECT_DIR, ".git")):
    !git -C {PROJECT_DIR} fetch origin && git -C {PROJECT_DIR} checkout {BRANCH} && git -C {PROJECT_DIR} reset --hard origin/{BRANCH}
else:
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    !git clone --branch {BRANCH} {GITHUB_REPO_URL} {PROJECT_DIR}

%cd /content/employeticsAD

print("\n=== 5) Installing Python Neural TTS & Audio Dependencies ===")
!pip install -q -r requirements.txt

print("\n=== 6) Installing Node / Remotion & Downloading Chrome for Testing (GPU mode) ===")
!npm install --no-audit --no-fund
!npx remotion gpu --chrome-mode=chrome-for-testing --gl=angle-egl --enable-multiprocess-on-linux || true
print("\nSetup complete!")


In [ ]:
#@title 3. Generate Studio Neural Voiceover + Soundtrack + UI SFX + Captions
%cd /content/employeticsAD
import os
from IPython.display import Audio, display

!python scripts/generate_audio.py \
  --engine {TTS_ENGINE} \
  --voice {KOKORO_VOICE} \
  --speed {VOICE_SPEED} \
  --music-volume {MUSIC_VOLUME}

assert os.path.exists("public/audio/master_audio.wav"), "master_audio.wav was not generated!"
assert os.path.exists("public/audio/voiceover.wav"), "voiceover.wav was not generated!"

print("\n1) Isolated Voiceover Preview:")
display(Audio("public/audio/voiceover.wav", autoplay=False))
print("\n2) Mastered Audio Track (Voiceover + Soundtrack + SFX):")
display(Audio("public/audio/master_audio.wav", autoplay=False))


In [ ]:
#@title 4. Render 40-Second Motion Graphics Commercial (Chrome GPU + NVIDIA NVENC + Audio Mux)
%cd /content/employeticsAD

caption_flag = "--captions" if BURN_CAPTIONS else ""
out_path = f"out/{OUTPUT_FILENAME}"

!python scripts/render_video.py \
  --output {out_path} \
  --scale {SCALE} \
  --crf 16 \
  {caption_flag}


In [ ]:
#@title 5. Preview Rendered Video Inline (with Audio)
from IPython.display import Video, display
import os

video_path = f"/content/employeticsAD/out/{OUTPUT_FILENAME}"
assert os.path.exists(video_path), f"Video not found at {video_path}"
print(f"Rendered video size: {os.path.getsize(video_path) / (1024*1024):.2f} MB")
display(Video(video_path, embed=True, width=960))


In [ ]:
#@title 6. Download MP4 + SRT Captions (and Optional Google Drive Backup)
import os, shutil
from google.colab import files

video_path = f"/content/employeticsAD/out/{OUTPUT_FILENAME}"
srt_path = "/content/employeticsAD/public/audio/captions.srt"

if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dest_dir = "/content/drive/MyDrive/EmployeticsAd"
    os.makedirs(dest_dir, exist_ok=True)
    shutil.copy2(video_path, os.path.join(dest_dir, OUTPUT_FILENAME))
    if os.path.exists(srt_path):
        shutil.copy2(srt_path, os.path.join(dest_dir, "captions.srt"))
    print(f"Saved copies to Google Drive: {dest_dir}")

files.download(video_path)
if os.path.exists(srt_path):
    files.download(srt_path)
